## Practical Exercise 3 — Uniform Cost Search (UCS)
 
**Problem:** Find the least-cost route for the autonomous trailer from Nairobi to Kisumu using the directed highway network developed in Part 1.  
**Cost measure:** Road distance in kilometres.




## 1. Problem formulation

| Element | Definition |
|---|---|
| Initial state | Nairobi |
| Goal state | Kisumu |
| States | Nairobi, Naivasha, Thika, Nyeri, Nanyuki, Nakuru, Kericho, Eldoret, Kisumu, Kisii, Kapsabet and Kitale |
| Actions | Traverse an outgoing directed road link from the current town to a connected town |
| Step cost | Distance of the selected road link, in kilometres |
| Path cost | Sum of the step costs from Nairobi to the current town |

The graph and distances below are taken from the group's Part 1 design. The search treats each listed connection as directed; it does not automatically assume a reverse road connection.

## 2. Directed highway graph

The adjacency list stores each town as a dictionary key. Its value is a list of `(destination, distance)` pairs. Only the outgoing links specified in Part 1 are included.

In [48]:
graph = {
    'Nairobi': [('Naivasha', 88), ('Thika', 42)],
    'Thika': [('Nyeri', 85)],
    'Nyeri': [('Nanyuki', 105)],
    'Nanyuki': [('Naivasha', 160)],
    'Naivasha': [('Nakuru', 68)],
    'Nakuru': [('Kericho', 110), ('Eldoret', 140)],
    'Kericho': [('Kisumu', 87), ('Kisii', 112)],
    'Kisii': [('Kisumu', 115)],
    'Eldoret': [('Kapsabet', 64), ('Kitale', 120)],
    'Kapsabet': [('Kisumu', 73)],
    'Kitale': [],
    'Kisumu': []
}

start = 'Nairobi'
goal = 'Kisumu'

print('Start:', start)
print('Goal:', goal)
print('Number of directed road links:', sum(len(edges) for edges in graph.values()))

Start: Nairobi
Goal: Kisumu
Number of directed road links: 14


### 2.1 Inspecting the graph

This cell prints every directed link and its distance. It provides a simple check that the graph contains the same 14 links as the Part 1 adjacency list.

In [49]:
for town, connections in graph.items():
    if connections:
        for destination, distance in connections:
            print(f'{town} -> {destination}: {distance} km')
    else:
        print(f'{town}: no outgoing links')

Nairobi -> Naivasha: 88 km
Nairobi -> Thika: 42 km
Thika -> Nyeri: 85 km
Nyeri -> Nanyuki: 105 km
Nanyuki -> Naivasha: 160 km
Naivasha -> Nakuru: 68 km
Nakuru -> Kericho: 110 km
Nakuru -> Eldoret: 140 km
Kericho -> Kisumu: 87 km
Kericho -> Kisii: 112 km
Kisii -> Kisumu: 115 km
Eldoret -> Kapsabet: 64 km
Eldoret -> Kitale: 120 km
Kapsabet -> Kisumu: 73 km
Kitale: no outgoing links
Kisumu: no outgoing links


## 3. Uniform Cost Search implementation

UCS uses a **priority queue** so that the route with the smallest accumulated cost is selected first. Python's `heapq` module provides this priority queue.

The implementation maintains:

- `frontier`: routes waiting to be explored, ordered by cumulative cost.
- `best_cost`: the lowest cost found so far for each town.
- `explored`: towns that have already been expanded.
- `path`: the sequence of towns used to reach a frontier entry.

The goal test is performed when a node is removed from the priority queue. With non-negative road distances, this is when UCS can safely return the optimal route.

In [50]:
import heapq
from itertools import count

def uniform_cost_search(graph, start, goal, show_trace=True):
    """Return the least-cost path from start to goal using Uniform Cost Search."""

    # A counter gives heap entries a stable tie-breaker when costs are equal.
    sequence = count()
    frontier = [(0, next(sequence), start, [start])]
    best_cost = {start: 0}
    explored = set()
    expansion_order = []

    if show_trace:
        print(f'Initial frontier: [({start}, cost=0)]')
        print('-' * 72)

    while frontier:
        cost, _, current, path = heapq.heappop(frontier)

        # Ignore an outdated queue entry if a cheaper route was found later.
        if cost != best_cost.get(current):
            continue

        # A town is expanded only once at its lowest known cost.
        if current in explored:
            continue

        # UCS checks the goal when it is popped, not when it is first generated.
        if current == goal:
            if show_trace:
                print(f'Goal reached: {current} at cumulative cost {cost} km')
            return {
                'path': path,
                'cost': cost,
                'expansion_order': expansion_order,
                'explored': explored
            }

        explored.add(current)
        expansion_order.append((current, cost))

        if show_trace:
            print(f'Expand {current} (cost={cost} km)')

        for neighbour, step_cost in graph.get(current, []):
            new_cost = cost + step_cost

            # Add a neighbour only if this route improves its best known cost.
            if neighbour not in explored and new_cost < best_cost.get(neighbour, float('inf')):
                best_cost[neighbour] = new_cost
                new_path = path + [neighbour]
                heapq.heappush(
                    frontier,
                    (new_cost, next(sequence), neighbour, new_path)
                )
                if show_trace:
                    print(f'  Add {neighbour}: {cost} + {step_cost} = {new_cost} km')

        if show_trace:
            pending = sorted((entry[0], entry[2]) for entry in frontier)
            print('  Frontier (lowest cost first):', pending)
            print()

    if show_trace:
        print('No route to the goal was found.')
    return None

## 4. Run UCS: Nairobi to Kisumu

The next cell executes the search on the graph. The trace shows which town is expanded, the cumulative cost at that point, the neighbours added to the frontier, and the frontier's cost order.

In [51]:
result = uniform_cost_search(graph, start, goal)

Initial frontier: [(Nairobi, cost=0)]
------------------------------------------------------------------------
Expand Nairobi (cost=0 km)
  Add Naivasha: 0 + 88 = 88 km
  Add Thika: 0 + 42 = 42 km
  Frontier (lowest cost first): [(42, 'Thika'), (88, 'Naivasha')]

Expand Thika (cost=42 km)
  Add Nyeri: 42 + 85 = 127 km
  Frontier (lowest cost first): [(88, 'Naivasha'), (127, 'Nyeri')]

Expand Naivasha (cost=88 km)
  Add Nakuru: 88 + 68 = 156 km
  Frontier (lowest cost first): [(127, 'Nyeri'), (156, 'Nakuru')]

Expand Nyeri (cost=127 km)
  Add Nanyuki: 127 + 105 = 232 km
  Frontier (lowest cost first): [(156, 'Nakuru'), (232, 'Nanyuki')]

Expand Nakuru (cost=156 km)
  Add Kericho: 156 + 110 = 266 km
  Add Eldoret: 156 + 140 = 296 km
  Frontier (lowest cost first): [(232, 'Nanyuki'), (266, 'Kericho'), (296, 'Eldoret')]

Expand Nanyuki (cost=232 km)
  Frontier (lowest cost first): [(266, 'Kericho'), (296, 'Eldoret')]

Expand Kericho (cost=266 km)
  Add Kisumu: 266 + 87 = 353 km
  Add Kisii

## 5. Optimal path and total cost

This cell presents the route returned by the search and calculates the individual road segments again from the graph. The segment check confirms that the reported total is the sum of the links in the returned path.

In [52]:
if result is not None:
    optimal_path = result['path']
    total_cost = result['cost']

    print('OPTIMAL ROUTE')
    print(' -> '.join(optimal_path))
    print('\nROAD SEGMENTS')

    calculated_total = 0
    for source, destination in zip(optimal_path, optimal_path[1:]):
        distance = next(
            distance for neighbour, distance in graph[source]
            if neighbour == destination
        )
        calculated_total += distance
        print(f'{source} -> {destination}: {distance} km')

    print('\nTotal path cost:', total_cost, 'km')
    print('Segment sum:', calculated_total, 'km')
    assert total_cost == calculated_total, 'The path cost does not match the segment sum.'
    print('Cost verification: PASSED')
else:
    print('No path was found.')

OPTIMAL ROUTE
Nairobi -> Naivasha -> Nakuru -> Kericho -> Kisumu

ROAD SEGMENTS
Nairobi -> Naivasha: 88 km
Naivasha -> Nakuru: 68 km
Nakuru -> Kericho: 110 km
Kericho -> Kisumu: 87 km

Total path cost: 353 km
Segment sum: 353 km
Cost verification: PASSED


## 6. Search expansion summary

The expansion order records the towns UCS expanded before removing the goal from the priority queue. The goal itself is detected when popped, so it is not added to the list of expanded non-goal towns.

In [53]:
print('Order of expanded towns:')
for number, (town, cost) in enumerate(result['expansion_order'], start=1):
    print(f'{number}. {town} — cumulative cost: {cost} km')

print('\nGoal town:', goal)
print('Goal path cost:', result['cost'], 'km')

Order of expanded towns:
1. Nairobi — cumulative cost: 0 km
2. Thika — cumulative cost: 42 km
3. Naivasha — cumulative cost: 88 km
4. Nyeri — cumulative cost: 127 km
5. Nakuru — cumulative cost: 156 km
6. Nanyuki — cumulative cost: 232 km
7. Kericho — cumulative cost: 266 km
8. Eldoret — cumulative cost: 296 km

Goal town: Kisumu
Goal path cost: 353 km


## 7. Verification against the Part 1 route calculations

The routes below are the alternatives calculated in the group's Part 1 design. Their costs can be checked against the result returned by UCS.

In [54]:
candidate_routes = {
    'Route A — Kericho corridor': ['Nairobi', 'Naivasha', 'Nakuru', 'Kericho', 'Kisumu'],
    'Route B — Kapsabet corridor': ['Nairobi', 'Naivasha', 'Nakuru', 'Eldoret', 'Kapsabet', 'Kisumu'],
    'Route C — Kisii circuit': ['Nairobi', 'Naivasha', 'Nakuru', 'Kericho', 'Kisii', 'Kisumu'],
    'Route D — Central Aberdare loop': ['Nairobi', 'Thika', 'Nyeri', 'Nanyuki', 'Naivasha', 'Nakuru', 'Kericho', 'Kisumu']
}

def calculate_route_cost(graph, route):
    total = 0
    for source, destination in zip(route, route[1:]):
        matching_edges = [cost for neighbour, cost in graph.get(source, []) if neighbour == destination]
        if not matching_edges:
            raise ValueError(f'No directed edge exists from {source} to {destination}.')
        total += matching_edges[0]
    return total

for route_name, route in candidate_routes.items():
    print(f'{route_name}: {calculate_route_cost(graph, route)} km')

print(f"\nUCS result: {' -> '.join(result['path'])} ({result['cost']} km)")

Route A — Kericho corridor: 353 km
Route B — Kapsabet corridor: 433 km
Route C — Kisii circuit: 493 km
Route D — Central Aberdare loop: 657 km

UCS result: Nairobi -> Naivasha -> Nakuru -> Kericho -> Kisumu (353 km)


## 8. Conclusion

Uniform Cost Search was applied to the directed highway graph from Nairobi to Kisumu, using road distance in kilometres as the path-cost measure. At each iteration, the algorithm selected the frontier entry with the smallest accumulated cost. It stopped when Kisumu was removed from the priority queue.

The search returned the route **Nairobi → Naivasha → Nakuru → Kericho → Kisumu**, with a total cost of **353 km**. This agrees with the route calculations in Part 1 and is the lowest-cost route among the alternatives represented in the graph.

The result is optimal for the supplied graph and edge costs. It represents the abstract search problem; it does not account for live traffic, road closures, weather, or other real-time driving conditions.